# Production-Compatible Extension of the Gurobi Itinerary Notebook

**Base notebook:** `notebook/itinerary_optimization_pipeline_extend_gurobi.ipynb`

This notebook is not a replacement for the original pipeline. It is a clean, GitHub-ready companion that keeps the same data-mining flow, variable names, and Gurobi optimization baseline, then adds publishable and production-facing extensions in staged sections.

Design rules:

- Preserve the original pipeline order: Yelp business data, OSM/Nominatim/Open-Meteo APIs, Yelp accommodation enrichment, travel matrices, XGBoost waiting-time model, live weather blending, Gurobi optimization.
- Use the original helper modules: `gurobi_itinerary_solver.py`, `itinerary_baseline.py`, `adaptive_bandit_planner.py`, and `sensitivity_analysis.py`.
- Do not replace Gurobi with a handmade route optimizer. Lightweight fallback cells are only for validation when Gurobi is unavailable.
- Mark future system layers as **[PROTOTYPE]**.

## 1. Contribution and System Targets

The notebook satisfies both research and industrial goals.

Research contributions:

1. A data-driven predictive optimization pipeline.
2. Quantitative comparison against ranking and greedy baselines.
3. A hierarchical optimization extension for multi-region / multi-city day allocation.
4. A hybrid bandit-optimization search layer where bandits find promising route bundles and Gurobi seeks feasible routes inside smaller subproblems.

Production proof of work:

- Backend: FastAPI.
- Model: the existing XGBoost waiting-time model.
- Database: PostgreSQL for requests, itineraries, and experiment logs.
- Deployment: AWS EC2 with Docker.
- Optional frontend: Streamlit first, React later.

## 2. Configuration

These flags make the notebook careful rather than monolithic. The default path uses local raw Yelp files if present. Live APIs and Gurobi are enabled through explicit flags so the same notebook can be used for research runs, GitHub review, and production prototyping.

In [1]:
import importlib
import json
import os
import sys
import time
from datetime import datetime
from pathlib import Path

import numpy as np
import pandas as pd
import requests

try:
    import xgboost as xgb

    XGBOOST_AVAILABLE = True
except ModuleNotFoundError:
    xgb = None
    XGBOOST_AVAILABLE = False

from geopy.distance import geodesic
from sklearn.ensemble import GradientBoostingRegressor
from sklearn.metrics import mean_squared_error
from sklearn.model_selection import train_test_split

RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

PROJECT_ROOT = Path.cwd()
if PROJECT_ROOT.name == "notebook":
    PROJECT_ROOT = PROJECT_ROOT.parent
NOTEBOOK_DIR = PROJECT_ROOT / "notebook"
SRC_DIR = PROJECT_ROOT / "src"
for module_dir in [NOTEBOOK_DIR, SRC_DIR]:
    if str(module_dir) not in sys.path:
        sys.path.insert(0, str(module_dir))

import itinerary_system.bandit_candidate_selector as bandit_candidate_selector_module
import itinerary_system.budget as budget_module
import itinerary_system.config as config_module
import itinerary_system.data_enrichment as data_enrichment_module
import itinerary_system.diversity as diversity_module
import itinerary_system.experiment_runner as experiment_runner_module
import itinerary_system.hierarchical_gurobi as hierarchical_gurobi_module
import itinerary_system.map_renderer as map_renderer_module
import itinerary_system.multi_objective_route as multi_objective_route_module
import itinerary_system.route_gurobi_oracle as route_gurobi_oracle_module
import itinerary_system.utility_model as utility_model_module

for _module in [
    config_module,
    utility_model_module,
    diversity_module,
    multi_objective_route_module,
    route_gurobi_oracle_module,
    data_enrichment_module,
    hierarchical_gurobi_module,
    budget_module,
    bandit_candidate_selector_module,
    experiment_runner_module,
    map_renderer_module,
]:
    importlib.reload(_module)

from itinerary_system.bandit_candidate_selector import (
    build_bandit_arms,
    route_search_strategies_from_config,
    run_bandit_gurobi_stress_benchmark,
    run_bandit_search,
)
from itinerary_system.budget import budget_bounds_from_df, estimate_budget_range
from itinerary_system.config import load_trip_config, save_resolved_config
from itinerary_system.data_enrichment import build_enriched_catalog, canonical_poi_columns
from itinerary_system.diversity import diversity_audit_row
from itinerary_system.experiment_runner import compare_trip_lengths
from itinerary_system.hierarchical_gurobi import candidate_plans, solve_hierarchical_trip_with_gurobi
from itinerary_system.route_gurobi_oracle import solve_enriched_route_with_gurobi

CONFIG_PATH = Path(os.getenv("TRIP_CONFIG_PATH", PROJECT_ROOT / "configs" / "default_trip_config.yaml"))
if not CONFIG_PATH.is_absolute():
    CONFIG_PATH = PROJECT_ROOT / CONFIG_PATH
NATURE_DEMO_CONFIG_PATH = PROJECT_ROOT / "configs" / "nature_trip_config.yaml"
config = load_trip_config(CONFIG_PATH)

DATASET_DIR = PROJECT_ROOT / "dataset"
OUTPUT_DIR = PROJECT_ROOT / "results" / "outputs"
FIGURE_DIR = PROJECT_ROOT / "results" / "figures"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)
save_resolved_config(config, OUTPUT_DIR / "production_resolved_config.json")

RAW_BUSINESS_PATH = Path(os.getenv("YELP_BUSINESS_JSON", DATASET_DIR / "yelp_academic_dataset_business.json"))
RAW_REVIEW_PATH = Path(os.getenv("YELP_REVIEW_JSON", DATASET_DIR / "yelp_academic_dataset_review.json"))
REVIEW_DENSITY_CACHE_PATH = Path(os.getenv("REVIEW_DENSITY_CACHE_PATH", OUTPUT_DIR / "production_review_density.csv"))

RUN_LIVE_APIS = bool(config.get("enrichment", "run_live_apis", False))
RUN_YELP_ACCOMMODATION_ENRICHMENT = (
    False  # Yelp live API is intentionally disabled; local academic Yelp data is used only.
)
RUN_GUROBI = os.getenv("RUN_GUROBI", "1") == "1"
RUN_GUROBI_SIZE_LIMITED_DEMO = os.getenv("RUN_GUROBI_SIZE_LIMITED_DEMO", "1") == "1"
RUN_ADAPTIVE_BANDIT = os.getenv("RUN_ADAPTIVE_BANDIT", "0") == "1"
RUN_FULL_REVIEW_SCAN = os.getenv("RUN_FULL_REVIEW_SCAN", "0") == "1"

CITY = "Santa Barbara"
COASTAL_CITIES = ["Santa Barbara", "Santa Cruz", "Monterey", "San Luis Obispo"]
CALIFORNIA_TRIP_CITIES = [
    "San Diego",
    "Los Angeles",
    "Santa Barbara",
    "San Luis Obispo",
    "Monterey",
    "Santa Cruz",
    "San Francisco",
]

print("Project root:", PROJECT_ROOT)
print("Config path:", CONFIG_PATH)
print("Business JSON exists:", RAW_BUSINESS_PATH.exists())
print("Review JSON exists:", RAW_REVIEW_PATH.exists())
print("RUN_LIVE_APIS:", RUN_LIVE_APIS)
print("RUN_GUROBI:", RUN_GUROBI)
print("RUN_GUROBI_SIZE_LIMITED_DEMO:", RUN_GUROBI_SIZE_LIMITED_DEMO)
print("RUN_FULL_REVIEW_SCAN:", RUN_FULL_REVIEW_SCAN)
display(config.to_frame())

Project root: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization
Config path: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/configs/default_trip_config.yaml
Business JSON exists: False
Review JSON exists: False
RUN_LIVE_APIS: False
RUN_GUROBI: True
RUN_GUROBI_SIZE_LIMITED_DEMO: True
RUN_FULL_REVIEW_SCAN: False


,section,parameter,value
0,trip,state,California
1,trip,trip_days,7
2,trip,start_city_options,"[San Francisco, Los Angeles]"
3,trip,end_city_options,"[Los Angeles, San Francisco]"
4,trip,traveler_profile,balanced
...,...,...,...
111,learning_to_rank,method,bpr
112,learning_to_rank,min_pairwise_examples,200
113,trip_duration_comparison,enabled,True
114,trip_duration_comparison,days,"[7, 9, 12]"


In [2]:
# for name in [
#     "production_method_comparison.csv",
#     "production_method_route_stops.csv",
#     "production_trip_length_comparison.csv",
#     "production_trip_length_route_stops.csv",
# ]:
#     path = OUTPUT_DIR / name
#     if path.exists():
#         path.unlink()
#         print("Deleted:", path)

# html_path = FIGURE_DIR / "production_hierarchical_trip_map.html"
# if html_path.exists():
#     html_path.unlink()
#     print("Deleted:", html_path)
import inspect

import blueprint_trip_map

import itinerary_system.map_renderer as map_renderer

print("blueprint_trip_map file:", blueprint_trip_map.__file__)
print("map_renderer file:", map_renderer.__file__)

print("\nDoes current imported blueprint have show_by_default?")
print("show_by_default" in inspect.getsource(blueprint_trip_map._add_model_comparison_layers))

print("\nDoes current imported blueprint hide scenic route by default?")
print("scenic=True,\n        show=False" in inspect.getsource(blueprint_trip_map.build_production_trip_map))

print("\nRoute-stop CSV checks:")
for name in [
    "production_method_route_stops.csv",
    "production_trip_length_route_stops.csv",
    "production_day_plan.csv",
]:
    path = OUTPUT_DIR / name
    print("\n", path)
    print("exists:", path.exists())
    if path.exists():
        df = pd.read_csv(path)
        print("shape:", df.shape)
        print("columns:", list(df.columns[:12]))
        if "method" in df.columns and "trip_days" in df.columns:
            print(df[["comparison_type", "comparison_label", "method", "trip_days"]].drop_duplicates().head(20))
        if {"latitude", "longitude"}.issubset(df.columns):
            print("valid coords:", df[["latitude", "longitude"]].dropna().shape[0])

blueprint_trip_map file: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/notebook/blueprint_trip_map.py
map_renderer file: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/src/itinerary_system/map_renderer.py

Does current imported blueprint have show_by_default?
True

Does current imported blueprint hide scenic route by default?
False

Route-stop CSV checks:

 /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/results/outputs/production_method_route_stops.csv
exists: True
shape: (49, 61)
columns: ['comparison_type', 'comparison_label', 'method', 'method_display_name', 'trip_days', 'gateway_start', 'gateway_end', 'day', 'stop_order', 'city', 'overnight_city', 'hotel_name']
   comparison_type                                   comparison_label  \
0           method  Method · Hierarchical + Bandit + Small Gurobi ...   
13          method              Method · Hierarchical Greed

## 3. Load Yelp Business Data Using the Original Notebook Logic

This cell follows Section 2.1 of `itinerary_optimization_pipeline_extend_gurobi.ipynb`. If the raw Yelp business file exists, we stream it directly. If not, the notebook falls back to the committed `coastal_attractions.csv` only for GitHub readability.

Why this matters: the baseline is still the original Yelp-derived utility pipeline, not a manually invented attraction table.

In [3]:
def load_yelp_business_attractions(raw_business_path, fallback_csv=None):
    records = []
    if raw_business_path.exists():
        with raw_business_path.open("r", encoding="utf-8") as handle:
            for line in handle:
                obj = json.loads(line)
                records.append(
                    {
                        "business_id": obj.get("business_id"),
                        "name": obj.get("name"),
                        "city": obj.get("city"),
                        "stars": obj.get("stars"),
                        "review_count": obj.get("review_count"),
                        "latitude": obj.get("latitude"),
                        "longitude": obj.get("longitude"),
                        "categories": obj.get("categories"),
                    }
                )
        source = "raw_yelp_business_json"
        attractions = pd.DataFrame(records)
    elif fallback_csv and Path(fallback_csv).exists():
        attractions = pd.read_csv(fallback_csv)
        if "business_id" not in attractions.columns:
            attractions["business_id"] = [f"fallback_{i:04d}" for i in range(len(attractions))]
        source = "committed_fallback_csv"
    else:
        raise FileNotFoundError("No Yelp business JSON or fallback attraction CSV found.")

    return attractions, source


attractions_df, attraction_source = load_yelp_business_attractions(
    RAW_BUSINESS_PATH,
    fallback_csv=OUTPUT_DIR / "coastal_attractions.csv",
)
all_business_df = attractions_df.copy()
print(f"Loaded {len(attractions_df):,} business rows from {attraction_source}")
attractions_df.head()

Loaded 10 business rows from committed_fallback_csv


,name,city,stars,review_count,latitude,longitude,categories,utility,business_id
0,Santa Barbara Zoo,Santa Barbara,4.0,681,34.419470,-119.668001,"Zoos, Event Planning & Services, Venues & Even...",26.100119,fallback_0000
1,Sustainable Wine Tours,Santa Barbara,5.0,358,34.422452,-119.705021,"Wine Tours, Hotels & Travel, Tours, Transporta...",29.416612,fallback_0001
2,Mission Santa Barbara,Santa Barbara,4.0,321,34.438201,-119.713556,"Churches, Arts & Entertainment, Museums, Relig...",23.098206,fallback_0002
3,Captain Jack's Tours & Events,Santa Barbara,4.5,310,34.420334,-119.710749,"Active Life, Wineries, Tours, Horseback Riding...",25.829068,fallback_0003
4,Stearns Wharf,Santa Barbara,4.0,297,34.410014,-119.685550,"Public Services & Government, Landmarks & Hist...",22.788374,fallback_0004


## 4. Filter Tourist Attractions and Build `top100`

This preserves the original variable names: `attractions_df` and `top100`. Keeping these names is important because the downstream Gurobi helper expects the same notebook context. The cell also declares the canonical attraction and city schemas used later by the production API and PostgreSQL design.

In [4]:
category_pattern = (
    "Museum|Museums|Park|Parks|Garden|Aquarium|Zoo|"
    "Landmark|Historic|Monument|Observatory|Tour|Attraction|"
    "Beach|Botanical"
)

attractions_df = attractions_df[attractions_df["city"].isin(COASTAL_CITIES)].copy()
attractions_df = attractions_df[attractions_df["categories"].notna()].copy()
attractions_df = attractions_df[
    attractions_df["categories"].str.contains(category_pattern, case=False, regex=True)
].copy()

attractions_df["review_count"] = pd.to_numeric(attractions_df["review_count"], errors="coerce").fillna(0)
attractions_df["stars"] = pd.to_numeric(attractions_df["stars"], errors="coerce").fillna(0)
attractions_df["utility"] = attractions_df["stars"] * np.log1p(attractions_df["review_count"])

top100 = attractions_df.sort_values("review_count", ascending=False).head(100).reset_index(drop=True)

CANONICAL_ATTRACTION_SCHEMA = {
    "business_id": "stable source identifier",
    "name": "attraction name",
    "city": "city name",
    "stars": "rating score",
    "review_count": "review volume / congestion proxy",
    "latitude": "WGS84 latitude",
    "longitude": "WGS84 longitude",
    "categories": "source category string",
    "utility": "rating-weighted utility score",
}
CANONICAL_CITY_SCHEMA = {
    "city": "city name",
    "candidate_attractions": "number of mined attraction candidates",
    "mean_utility": "mean catalog utility",
    "total_reviews": "aggregate review volume",
    "data_status": "yelp_sufficient / yelp_sparse / fallback needed",
    "data_quality_score": "0-1 readiness score for optimization",
}
canonical_missing = [column for column in CANONICAL_ATTRACTION_SCHEMA if column not in top100.columns]
assert not canonical_missing, f"Missing canonical attraction columns: {canonical_missing}"

assert not top100.empty, "top100 is empty; check city/category filters."
with (OUTPUT_DIR / "canonical_attraction_schema.json").open("w", encoding="utf-8") as handle:
    json.dump(CANONICAL_ATTRACTION_SCHEMA, handle, indent=2)
with (OUTPUT_DIR / "canonical_city_schema.json").open("w", encoding="utf-8") as handle:
    json.dump(CANONICAL_CITY_SCHEMA, handle, indent=2)
print(f"Filtered tourist attractions: {len(attractions_df):,}")
print(f"Selected top100 attractions: {len(top100):,}")
top100[["name", "city", "stars", "review_count", "utility"]].head(10)

Filtered tourist attractions: 10
Selected top100 attractions: 10


,name,city,stars,review_count,utility
0,Santa Barbara Zoo,Santa Barbara,4.0,681,26.100119
1,Sustainable Wine Tours,Santa Barbara,5.0,358,29.416612
2,Mission Santa Barbara,Santa Barbara,4.0,321,23.098206
3,Captain Jack's Tours & Events,Santa Barbara,4.5,310,25.829068
4,Stearns Wharf,Santa Barbara,4.0,297,22.788374
5,Celebration Cruises,Santa Barbara,4.5,297,25.636921
6,Santa Barbara Botanic Garden,Santa Barbara,4.0,266,22.348995
7,Santa Barbara County Courthouse,Santa Barbara,4.5,251,24.882431
8,Arroyo Burro Beach,Santa Barbara,4.5,251,24.882431
9,SB Buggie,Santa Barbara,5.0,231,27.233687


## 5. Data Mining APIs: City Coordinates, Hotels, and Historical Weather

This section keeps the prior API path:

- Nominatim for city coordinates.
- OpenStreetMap Overpass for accommodations.
- Open-Meteo archive for historical weather.

Live calls run only when `RUN_LIVE_APIS=1`. Otherwise, the notebook uses committed outputs or robust fallback data. This keeps GitHub execution stable while preserving the real data-mining design.

In [5]:
headers = {"User-Agent": "IE5533-WeatherAwareTravelPlanner/1.0"}


def get_city_coordinates(city, run_live=True):
    if run_live:
        geo_url = "https://nominatim.openstreetmap.org/search"
        geo_params = {"q": city, "format": "json", "limit": 1}
        response = requests.get(geo_url, params=geo_params, headers=headers, timeout=30)
        response.raise_for_status()
        result = response.json()[0]
        return float(result["lat"]), float(result["lon"]), "nominatim"

    if city == "Santa Barbara":
        return 34.4208, -119.6982, "static_santa_barbara_coordinate"
    raise ValueError(f"No offline coordinate fallback for {city}")


lat, lon, coordinate_source = get_city_coordinates(CITY, run_live=RUN_LIVE_APIS)
print(f"City: {CITY} | Coordinates: ({lat:.4f}, {lon:.4f}) | Source: {coordinate_source}")

City: Santa Barbara | Coordinates: (34.4208, -119.6982) | Source: static_santa_barbara_coordinate


In [6]:
def fetch_osm_accommodations(city_lat, city_lon, radius=10000, run_live=True):
    fallback_elements = [
        {"lat": 34.4208, "lon": -119.6982, "tags": {"name": "Harbor View Inn", "tourism": "hotel", "stars": "4"}},
        {"lat": 34.4260, "lon": -119.6940, "tags": {"name": "Santa Barbara Inn", "tourism": "hotel", "stars": "4"}},
        {
            "lat": 34.4380,
            "lon": -119.6850,
            "tags": {"name": "Simpson House Inn", "tourism": "guest_house", "stars": "4"},
        },
        {"lat": 34.4100, "lon": -119.7050, "tags": {"name": "Hostel Santa Barbara", "tourism": "hostel", "stars": "3"}},
        {"lat": 34.4285, "lon": -119.7170, "tags": {"name": "Sunset Motel", "tourism": "motel", "stars": "2"}},
        {
            "lat": 34.4165,
            "lon": -119.6922,
            "tags": {"name": "Downtown Garden Suites", "tourism": "apartment", "stars": "4"},
        },
    ]

    if run_live:
        query = f"""
        [out:json];
        (
          node["tourism"~"hotel|guest_house|hostel|motel|apartment"](around:{radius},{city_lat},{city_lon});
          way["tourism"~"hotel|guest_house|hostel|motel|apartment"](around:{radius},{city_lat},{city_lon});
          relation["tourism"~"hotel|guest_house|hostel|motel|apartment"](around:{radius},{city_lat},{city_lon});
        );
        out center;
        """
        url = "https://overpass-api.de/api/interpreter"
        for attempt in range(3):
            try:
                response = requests.get(url, params={"data": query}, timeout=30)
                response.raise_for_status()
                return response.json()["elements"], "overpass"
            except Exception as exc:
                print(f"Overpass attempt {attempt + 1}/3 failed: {type(exc).__name__}")
                time.sleep(3)

    return fallback_elements, "fallback_accommodations"


hotel_elements, hotel_source = fetch_osm_accommodations(lat, lon, run_live=RUN_LIVE_APIS)
hotels = []
for element in hotel_elements:
    tags = element.get("tags", {})
    element_lat = element.get("lat", element.get("center", {}).get("lat"))
    element_lon = element.get("lon", element.get("center", {}).get("lon"))
    if element_lat is None or element_lon is None:
        continue
    hotels.append(
        {
            "name": tags.get("name", "unknown"),
            "type": tags.get("tourism", "hotel"),
            "stars": tags.get("stars"),
            "latitude": float(element_lat),
            "longitude": float(element_lon),
        }
    )

hotels_df = pd.DataFrame(hotels).drop_duplicates(subset=["name", "latitude", "longitude"]).reset_index(drop=True)
print(f"Accommodation source: {hotel_source} | rows: {len(hotels_df)}")
hotels_df.head()

Accommodation source: fallback_accommodations | rows: 6


,name,type,stars,latitude,longitude
0,Harbor View Inn,hotel,4,34.4208,-119.6982
1,Santa Barbara Inn,hotel,4,34.4260,-119.6940
2,Simpson House Inn,guest_house,4,34.4380,-119.6850
3,Hostel Santa Barbara,hostel,3,34.4100,-119.7050
4,Sunset Motel,motel,2,34.4285,-119.7170


In [7]:
def load_or_fetch_weather(city_lat, city_lon, run_live=True):
    committed_weather = OUTPUT_DIR / "santa_barbara_weather.csv"
    if not run_live and committed_weather.exists():
        weather = pd.read_csv(committed_weather)
        weather["date"] = pd.to_datetime(weather["date"]).dt.date
        return weather, "committed_weather_csv"

    weather_url = "https://archive-api.open-meteo.com/v1/archive"
    weather_params = {
        "latitude": city_lat,
        "longitude": city_lon,
        "start_date": "2001-01-01",
        "end_date": "2026-03-01",
        "daily": "temperature_2m_max,temperature_2m_min,precipitation_sum",
        "timezone": "auto",
    }
    response = requests.get(weather_url, params=weather_params, timeout=60)
    response.raise_for_status()
    weather_json = response.json()
    weather = pd.DataFrame(
        {
            "date": weather_json["daily"]["time"],
            "temp_max": weather_json["daily"]["temperature_2m_max"],
            "temp_min": weather_json["daily"]["temperature_2m_min"],
            "precipitation": weather_json["daily"]["precipitation_sum"],
        }
    )
    weather["date"] = pd.to_datetime(weather["date"]).dt.date
    weather["avg_temp"] = (weather["temp_max"] + weather["temp_min"]) / 2
    weather["RainFlag"] = (weather["precipitation"] > 0).astype(int)
    weather["ColdFlag"] = (weather["avg_temp"] < 5).astype(int)
    weather["day_of_week"] = pd.to_datetime(weather["date"]).dt.dayofweek
    weather["Weekend"] = (weather["day_of_week"] >= 5).astype(int)
    weather.to_csv(OUTPUT_DIR / "santa_barbara_weather.csv", index=False)
    return weather, "open_meteo_archive"


weather_df, weather_source = load_or_fetch_weather(lat, lon, run_live=RUN_LIVE_APIS)
print(f"Weather source: {weather_source} | rows: {len(weather_df):,}")
weather_df.head()

Weather source: committed_weather_csv | rows: 9,191


,date,temp_max,temp_min,precipitation,avg_temp,RainFlag,ColdFlag,day_of_week,Weekend
0,2001-01-01,20.9,8.3,0.0,14.60,0,0,0,0
1,2001-01-02,24.8,8.2,0.0,16.50,0,0,1,0
2,2001-01-03,25.1,8.5,0.0,16.80,0,0,2,0
3,2001-01-04,24.3,8.0,0.0,16.15,0,0,3,0
4,2001-01-05,20.3,9.5,0.0,14.90,0,0,4,0


## 6. Open-Data Hotel Price and Rating Proxies

No live Yelp API is required. OSM/project hotel candidates are converted into the columns required by `gurobi_itinerary_solver.py`: `nightly_price`, `rating_score`, `experience_score`, and `name`. Prices are transparent planning proxies, not booking quotes.


In [8]:
TYPE_PRICE_DEFAULTS = {
    "hotel": 145.0,
    "hostel": 55.0,
    "guest_house": 125.0,
    "motel": 85.0,
    "airbnb": 135.0,
    "apartment": 135.0,
}
TYPE_RATING_DEFAULTS = {"hotel": 4.1, "hostel": 3.5, "guest_house": 4.3, "motel": 3.2, "airbnb": 4.2, "apartment": 4.2}
TYPE_EXPERIENCE_DEFAULTS = {
    "hotel": 4.0,
    "hostel": 3.3,
    "guest_house": 4.2,
    "motel": 3.0,
    "airbnb": 4.1,
    "apartment": 4.1,
}


def review_strength(review_count):
    return min(np.log1p(float(review_count)) / np.log1p(500), 1.0)


def estimate_price_from_type(row):
    lodging_type = row.get("lodging_type", row.get("type", "hotel"))
    base_price = TYPE_PRICE_DEFAULTS.get(lodging_type, TYPE_PRICE_DEFAULTS["hotel"])
    stars = row["stars"] if pd.notna(row["stars"]) else TYPE_RATING_DEFAULTS.get(lodging_type, 4.0)
    return round(base_price + 10.0 * max(float(stars) - 3.0, 0), 2)


def build_experience_score(rating_score, review_count, lodging_type):
    if pd.isna(rating_score):
        return TYPE_EXPERIENCE_DEFAULTS.get(lodging_type, 4.0)
    confidence_component = 1.0 + 4.0 * review_strength(review_count)
    return round(0.65 * float(rating_score) + 0.35 * confidence_component, 2)


hotels_df = hotels_df.copy()
hotels_df["lodging_type"] = hotels_df["type"].fillna("hotel").str.lower().replace({"guest house": "guest_house"})
hotels_df["stars"] = pd.to_numeric(hotels_df.get("stars"), errors="coerce")
hotels_df["price_tier"] = "estimated"
hotels_df["nightly_price"] = hotels_df.apply(estimate_price_from_type, axis=1)
hotels_df["price_source"] = "osm_type_estimate"
hotels_df["price_estimated"] = True
hotels_df["review_count"] = 0
hotels_df["rating_score"] = hotels_df.apply(
    lambda row: row["stars"] if pd.notna(row["stars"]) else TYPE_RATING_DEFAULTS.get(row["lodging_type"], 4.0),
    axis=1,
)
hotels_df["experience_score"] = hotels_df.apply(
    lambda row: build_experience_score(row["rating_score"], row["review_count"], row["lodging_type"]),
    axis=1,
)
hotels_df["value_score"] = (hotels_df["rating_score"] * hotels_df["experience_score"]) / hotels_df[
    "nightly_price"
].clip(lower=1)
hotels_df = hotels_df.sort_values(["nightly_price", "rating_score"], ascending=[True, False]).reset_index(drop=True)

required_hotel_columns = ["nightly_price", "rating_score", "experience_score", "name"]
missing_hotel_columns = [
    column for column in required_hotel_columns if column not in hotels_df.columns or hotels_df[column].isna().any()
]
if missing_hotel_columns:
    raise ValueError(f"Hotel data missing Gurobi-required columns after open-data enrichment: {missing_hotel_columns}")

print("Live Yelp API disabled: using OSM/project hotel candidates with transparent price proxies.")
hotels_df[
    [
        "name",
        "lodging_type",
        "price_tier",
        "nightly_price",
        "rating_score",
        "review_count",
        "experience_score",
        "price_source",
    ]
].head()

Live Yelp API disabled: using OSM/project hotel candidates with transparent price proxies.


,name,lodging_type,price_tier,nightly_price,rating_score,review_count,experience_score,price_source
0,Hostel Santa Barbara,hostel,estimated,55.0,3,0,2.30,osm_type_estimate
1,Sunset Motel,motel,estimated,85.0,2,0,1.65,osm_type_estimate
2,Simpson House Inn,guest_house,estimated,135.0,4,0,2.95,osm_type_estimate
3,Downtown Garden Suites,apartment,estimated,145.0,4,0,2.95,osm_type_estimate
4,Harbor View Inn,hotel,estimated,155.0,4,0,2.95,osm_type_estimate


## 7. Travel Matrices

This preserves the original travel matrix variables used by the Gurobi helper:

- `distance_matrix`
- `travel_time_matrix`
- `hotel_to_attr`
- `hotel_to_attr_time`
- `hotel_to_hotel`
- `hotel_to_hotel_time`

In [9]:
locations = top100[["name", "latitude", "longitude"]].reset_index(drop=True)
n = len(locations)

distance_matrix = np.zeros((n, n))
for i in range(n):
    for j in range(i, n):
        coord_i = (locations.loc[i, "latitude"], locations.loc[i, "longitude"])
        coord_j = (locations.loc[j, "latitude"], locations.loc[j, "longitude"])
        distance = geodesic(coord_i, coord_j).km
        distance_matrix[i, j] = distance
        distance_matrix[j, i] = distance

avg_speed = 20
travel_time_matrix = distance_matrix / avg_speed * 60
travel_time_df = pd.DataFrame(travel_time_matrix, index=locations["name"], columns=locations["name"])
travel_time_df.to_csv(OUTPUT_DIR / "travel_time_matrix.csv")

hotel_locations = hotels_df[["latitude", "longitude"]].values
attraction_locations = top100[["latitude", "longitude"]].values

num_hotels = len(hotel_locations)
num_attr = len(attraction_locations)

hotel_to_attr = np.zeros((num_hotels, num_attr))
for h in range(num_hotels):
    for i in range(num_attr):
        hotel_to_attr[h, i] = geodesic(tuple(hotel_locations[h]), tuple(attraction_locations[i])).km
hotel_to_attr_time = hotel_to_attr / avg_speed * 60

hotel_to_hotel = np.zeros((num_hotels, num_hotels))
for h in range(num_hotels):
    for g in range(num_hotels):
        hotel_to_hotel[h, g] = geodesic(tuple(hotel_locations[h]), tuple(hotel_locations[g])).km
hotel_to_hotel_time = hotel_to_hotel / avg_speed * 60

hotels_df.to_csv(OUTPUT_DIR / "hotel_dataset.csv", index=False)
pd.DataFrame(hotel_to_hotel_time, index=hotels_df["name"], columns=hotels_df["name"]).to_csv(
    OUTPUT_DIR / "hotel_to_hotel_time.csv"
)

print("Travel matrix:", travel_time_matrix.shape)
print("Hotel-to-attraction matrix:", hotel_to_attr_time.shape)
print("Hotel-to-hotel matrix:", hotel_to_hotel_time.shape)

Travel matrix: (10, 10)
Hotel-to-attraction matrix: (6, 10)
Hotel-to-hotel matrix: (6, 6)


## 8. Review Density and Waiting-Time Training Data

This cell keeps the original review-density logic, but makes the notebook runnable. For full research runs, set `RUN_FULL_REVIEW_SCAN=1` to stream the raw Yelp review JSON and cache matched review-density rows. For GitHub/demo runs, the notebook uses the cache if available, then falls back to a deterministic review-density proxy derived from Yelp business review counts and weather dates.

In [10]:
selected_ids = set(top100["business_id"].astype(str))


def build_review_density_proxy(top100_frame, weather_frame):
    """Data-derived fallback when the 5GB raw Yelp review file is unavailable or intentionally skipped."""
    sample_dates = weather_frame[["date", "RainFlag", "Weekend"]].drop_duplicates("date").copy()
    sample_dates = sample_dates.sort_values("date").iloc[::30].head(365)
    if sample_dates.empty:
        raise ValueError("Cannot build review-density proxy without weather dates.")

    rows = []
    for _, attr in top100_frame.iterrows():
        base_daily_reviews = max(float(attr.get("review_count", 0.0)) / max(len(sample_dates), 1), 0.05)
        for _, day in sample_dates.iterrows():
            weekend_multiplier = 1.25 if int(day.get("Weekend", 0)) else 0.95
            rain_multiplier = 0.85 if int(day.get("RainFlag", 0)) else 1.05
            proxy_reviews = max(base_daily_reviews * weekend_multiplier * rain_multiplier, 0.01)
            rows.append(
                {
                    "business_id": attr["business_id"],
                    "date": day["date"],
                    "review_count": proxy_reviews,
                    "density_source": "business_review_count_weather_proxy",
                }
            )
    density = pd.DataFrame(rows)
    review_df = density.loc[density.index.repeat(np.ceil(density["review_count"]).astype(int).clip(1, 20))]
    review_df = review_df[["business_id", "date"]].reset_index(drop=True)
    return review_df, density


def load_review_density(
    raw_review_path, selected_business_ids, top100_frame, weather_frame, cache_path, run_full_scan=False
):
    if cache_path.exists() and not run_full_scan:
        density = pd.read_csv(cache_path)
        density["date"] = pd.to_datetime(density["date"]).dt.date
        density = density[density["business_id"].astype(str).isin(selected_business_ids)].copy()
        if not density.empty:
            review_df = density.loc[density.index.repeat(np.ceil(density["review_count"]).astype(int).clip(1, 20))]
            review_df = review_df[["business_id", "date"]].reset_index(drop=True)
            density["density_source"] = density.get("density_source", "cached_review_density")
            return review_df, density, "cached_review_density"

    if raw_review_path.exists() and run_full_scan:
        reviews = []
        with raw_review_path.open("r", encoding="utf-8") as handle:
            for row_idx, line in enumerate(handle, start=1):
                obj = json.loads(line)
                business_id = obj.get("business_id")
                if business_id in selected_business_ids:
                    reviews.append({"business_id": business_id, "date": obj.get("date")})
                if row_idx % 1_000_000 == 0:
                    print(f"Scanned {row_idx:,} reviews | matched {len(reviews):,}")

        review_df = pd.DataFrame(reviews)
        if review_df.empty:
            print("No matching raw reviews found; using data-derived proxy.")
            return (*build_review_density_proxy(top100_frame, weather_frame), "business_review_count_weather_proxy")
        review_df["date"] = pd.to_datetime(review_df["date"]).dt.date
        density = review_df.groupby(["business_id", "date"]).size().reset_index(name="review_count")
        density["density_source"] = "raw_yelp_review_json"
        density.to_csv(cache_path, index=False)
        return review_df, density, "raw_yelp_review_json"

    review_df, density = build_review_density_proxy(top100_frame, weather_frame)
    return review_df, density, "business_review_count_weather_proxy"


review_df, review_density, review_density_source = load_review_density(
    RAW_REVIEW_PATH,
    selected_ids,
    top100,
    weather_df,
    REVIEW_DENSITY_CACHE_PATH,
    run_full_scan=RUN_FULL_REVIEW_SCAN,
)
print("Review-density source:", review_density_source)
print(f"Matched/proxy reviews: {len(review_df):,}")
print(f"Review-density rows: {len(review_density):,}")
review_density.head()

Review-density source: business_review_count_weather_proxy
Matched/proxy reviews: 4,840
Review-density rows: 3,070


,business_id,date,review_count,density_source
0,fallback_0000,2001-01-01,2.212695,business_review_count_weather_proxy
1,fallback_0000,2001-01-31,2.212695,business_review_count_weather_proxy
2,fallback_0000,2001-03-02,2.212695,business_review_count_weather_proxy
3,fallback_0000,2001-04-01,2.911441,business_review_count_weather_proxy
4,fallback_0000,2001-05-01,2.212695,business_review_count_weather_proxy


## 9. Visit Duration Estimation

This preserves the PERT-style duration estimation from the original notebook and creates `visit_duration_sim`, a required input for the Gurobi optimizer.

In [11]:
DURATION_MAP = {
    "Zoo": (90, 120, 180),
    "Museum": (60, 90, 120),
    "Aquarium": (90, 120, 150),
    "Botanical": (60, 90, 120),
    "Garden": (60, 90, 120),
    "Park": (60, 90, 120),
    "Beach": (60, 90, 120),
    "Historic": (45, 60, 90),
    "Landmark": (45, 60, 75),
    "Monument": (45, 60, 75),
    "Observatory": (60, 90, 120),
    "Tour": (100, 120, 180),
    "Attraction": (60, 90, 120),
}


def pert_mean(a, m, b):
    return (a + 4 * m + b) / 6


def pert_sd(a, b):
    return (b - a) / 6


def estimate_duration(category_string):
    category_string = str(category_string)
    for key in DURATION_MAP:
        if key.lower() in category_string.lower():
            a, m, b = DURATION_MAP[key]
            return pert_mean(a, m, b)
    return 60


def estimate_sigma(category_string):
    category_string = str(category_string)
    for key in DURATION_MAP:
        if key.lower() in category_string.lower():
            a, _, b = DURATION_MAP[key]
            return pert_sd(a, b)
    return 15


top100["estimated_duration"] = top100["categories"].apply(estimate_duration)
top100["sigma"] = top100["categories"].apply(estimate_sigma)
top100["visit_duration_sim"] = rng.normal(top100["estimated_duration"], top100["sigma"]).clip(30, 240)

top100[["name", "estimated_duration", "sigma", "visit_duration_sim"]].to_csv(
    OUTPUT_DIR / "attraction_durations.csv", index=False
)
top100[["name", "categories", "estimated_duration", "sigma", "visit_duration_sim"]].head()

,name,categories,estimated_duration,sigma,visit_duration_sim
0,Santa Barbara Zoo,"Zoos, Event Planning & Services, Venues & Even...",125.000000,15.000000,129.570756
1,Sustainable Wine Tours,"Wine Tours, Hotels & Travel, Tours, Transporta...",126.666667,13.333333,112.800212
2,Mission Santa Barbara,"Churches, Arts & Entertainment, Museums, Relig...",90.000000,10.000000,97.504512
3,Captain Jack's Tours & Events,"Active Life, Wineries, Tours, Horseback Riding...",126.666667,13.333333,139.207530
4,Stearns Wharf,"Public Services & Government, Landmarks & Hist...",62.500000,7.500000,47.867236


## 10. XGBoost Waiting-Time Model

This is the official ML model in the project. It follows the original features:

```text
day_of_week, month, weekend, RainFlag -> review_count density
```

The predicted review density is converted to estimated visitors using the original participation-rate assumption, then transformed into waiting time by attraction type and capacity.

In [12]:
review_weather_density = review_density.merge(
    weather_df[["date", "RainFlag", "ColdFlag"]],
    on="date",
    how="left",
)
review_weather_density["day_of_week"] = pd.to_datetime(review_weather_density["date"]).dt.dayofweek
review_weather_density["month"] = pd.to_datetime(review_weather_density["date"]).dt.month
review_weather_density["weekend"] = (review_weather_density["day_of_week"] >= 5).astype(int)
review_weather_density[["RainFlag", "ColdFlag"]] = review_weather_density[["RainFlag", "ColdFlag"]].fillna(0)

X = review_weather_density[["day_of_week", "month", "weekend", "RainFlag"]]
y = review_weather_density["review_count"]

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=RANDOM_SEED)
if XGBOOST_AVAILABLE:
    model = xgb.XGBRegressor(
        objective="reg:squarederror",
        n_estimators=200,
        max_depth=6,
        learning_rate=0.05,
        random_state=RANDOM_SEED,
    )
    waiting_time_model_name = "XGBoost"
else:
    model = GradientBoostingRegressor(
        n_estimators=200,
        max_depth=3,
        learning_rate=0.05,
        random_state=RANDOM_SEED,
    )
    waiting_time_model_name = "GradientBoosting fallback"
model.fit(X_train, y_train)

y_pred = model.predict(X_test)
mse = mean_squared_error(y_test, y_pred)
rmse = float(np.sqrt(mse))
print(f"{waiting_time_model_name} review-density RMSE: {rmse:.3f}")

review_weather_density["predicted_review_count"] = model.predict(X)
PARTICIPATION_RATE = 0.00078
review_weather_density["estimated_visitors"] = review_weather_density["predicted_review_count"] / PARTICIPATION_RATE

XGBoost review-density RMSE: 0.459


In [13]:
def classify_attraction(name):
    name_lower = str(name).lower()
    if "beach" in name_lower:
        return "beach"
    if "park" in name_lower:
        return "park"
    if "museum" in name_lower:
        return "museum"
    if "zoo" in name_lower:
        return "zoo"
    if any(token in name_lower for token in ["wine", "whale", "tour"]):
        return "tour"
    if "adventure" in name_lower:
        return "activity"
    return "general"


top100["type"] = top100["name"].apply(classify_attraction)
review_weather_density = review_weather_density.merge(top100[["business_id", "type"]], on="business_id", how="left")

CAPACITY_TABLE = {"beach": 5000, "park": 3000, "museum": 800, "zoo": 700, "tour": 200, "activity": 150, "general": 500}
BASE_WAIT_TABLE = {"beach": 8, "park": 6, "museum": 15, "zoo": 25, "tour": 20, "activity": 14, "general": 12}


def compute_waiting(row):
    attraction_type = row.get("type", "general")
    capacity = CAPACITY_TABLE.get(attraction_type, CAPACITY_TABLE["general"])
    base_wait = BASE_WAIT_TABLE.get(attraction_type, BASE_WAIT_TABLE["general"])
    visitors = row.get("estimated_visitors", 0)
    wait = base_wait * np.log1p(visitors / capacity)
    if row.get("RainFlag", 0) and attraction_type in {"beach", "park", "activity"}:
        wait *= 0.85
    elif row.get("RainFlag", 0) and attraction_type in {"museum", "zoo"}:
        wait *= 1.10
    return wait


review_weather_density["waiting_time"] = review_weather_density.apply(compute_waiting, axis=1).clip(1, 120)
waiting_time_avg = review_weather_density.groupby("business_id")["waiting_time"].mean().reset_index()

top100_with_waiting_time = top100.merge(waiting_time_avg, on="business_id", how="left")
top100_with_waiting_time["waiting_time"] = top100_with_waiting_time["waiting_time"].fillna(
    top100_with_waiting_time["waiting_time"].median()
)

top100_with_waiting_time[["name", "type", "waiting_time"]].head()

,name,type,waiting_time
0,Santa Barbara Zoo,zoo,28.047455
1,Sustainable Wine Tours,tour,41.685240
2,Mission Santa Barbara,general,16.083166
3,Captain Jack's Tours & Events,tour,41.685240
4,Stearns Wharf,general,16.083166


## 11. Live Weather Blending

This preserves the original real-time weather blending idea. If `RUN_LIVE_APIS=0`, it uses the latest historical weather row as a deterministic GitHub-safe substitute.

In [14]:
def get_live_weather_context(city_lat, city_lon, run_live=True):
    now = datetime.now()
    if run_live:
        weather_url = "https://api.open-meteo.com/v1/forecast"
        weather_params = {
            "latitude": city_lat,
            "longitude": city_lon,
            "current_weather": True,
            "hourly": "precipitation_probability",
            "forecast_days": 1,
        }
        response = requests.get(weather_url, params=weather_params, timeout=30)
        response.raise_for_status()
        weather_now = response.json()
        rain_probability = float(weather_now["hourly"]["precipitation_probability"][0])
        return (
            now.weekday(),
            now.month,
            int(now.weekday() >= 5),
            int(rain_probability > 50),
            rain_probability,
            "open_meteo_forecast",
        )

    latest = weather_df.iloc[-1]
    return (
        int(latest["day_of_week"]),
        int(pd.to_datetime(latest["date"]).month),
        int(latest["Weekend"]),
        int(latest["RainFlag"]),
        float(latest["precipitation"]),
        "historical_latest_row",
    )


day_of_week, month, weekend, RainFlag, rain_probability, live_weather_source = get_live_weather_context(
    lat, lon, RUN_LIVE_APIS
)
X_live = pd.DataFrame({"day_of_week": [day_of_week], "month": [month], "weekend": [weekend], "RainFlag": [RainFlag]})

pred_review_density = model.predict(X_live)[0]
estimated_visitors_live = pred_review_density / PARTICIPATION_RATE

live_waiting = []
for attraction_type in top100_with_waiting_time["type"]:
    capacity = CAPACITY_TABLE.get(attraction_type, 500)
    base_wait = BASE_WAIT_TABLE.get(attraction_type, 10)
    wait = base_wait * np.log1p(estimated_visitors_live / capacity)
    live_waiting.append(wait)

top100_with_waiting_time["waiting_time_live"] = live_waiting
top100_with_waiting_time["waiting_final"] = (
    0.7 * top100_with_waiting_time["waiting_time"] + 0.3 * top100_with_waiting_time["waiting_time_live"]
)

print("Live weather source:", live_weather_source)
print(
    f"Waiting range: {top100_with_waiting_time['waiting_final'].min():.1f} - {top100_with_waiting_time['waiting_final'].max():.1f} minutes"
)

Live weather source: historical_latest_row
Waiting range: 2.1 - 43.2 minutes


## 12. Cost Estimation

The Gurobi helper expects a `cost` column. This follows the original type-based cost approach.

In [15]:
COST_TABLE = {"beach": 0, "park": 5, "museum": 18, "zoo": 30, "tour": 55, "activity": 45, "general": 20}
top100_with_waiting_time["cost"] = top100_with_waiting_time["type"].map(COST_TABLE).fillna(20)

required_gurobi_columns = ["utility", "waiting_final", "visit_duration_sim", "cost", "name"]
missing_gurobi_columns = [
    column for column in required_gurobi_columns if column not in top100_with_waiting_time.columns
]
assert not missing_gurobi_columns, f"Missing Gurobi input columns: {missing_gurobi_columns}"

top100_with_waiting_time[["name", "type", "utility", "waiting_final", "visit_duration_sim", "cost"]].head()

,name,type,utility,waiting_final,visit_duration_sim,cost
0,Santa Barbara Zoo,zoo,26.100119,29.357512,129.570756,30
1,Sustainable Wine Tours,tour,29.416612,43.171713,112.800212,55
2,Mission Santa Barbara,general,23.098206,16.844300,97.504512,20
3,Captain Jack's Tours & Events,tour,25.829068,43.171713,139.207530,55
4,Stearns Wharf,general,22.788374,16.844300,47.867236,20


## 13. Open-Data Multi-Source Enrichment

This is the first optimization-facing data layer. Yelp is local/offline only; live enrichment uses free/open sources such as OSM, Wikidata, Wikipedia, Open-Meteo, and OSRM caches when enabled.


In [16]:
_enrichment_outputs = build_enriched_catalog(
    all_business_df=all_business_df,
    city_names=CALIFORNIA_TRIP_CITIES,
    output_dir=OUTPUT_DIR,
    config=config,
)

osm_city_poi_catalog_df = _enrichment_outputs["osm_city_poi_catalog_df"]
osm_city_hotel_catalog_df = _enrichment_outputs["osm_city_hotel_catalog_df"]
production_enriched_poi_catalog_df = _enrichment_outputs["production_enriched_poi_catalog_df"]
production_enrichment_audit_df = _enrichment_outputs["production_enrichment_audit_df"]
city_catalog_summary_df = _enrichment_outputs["city_catalog_summary_df"]
social_signal_snapshots_df = _enrichment_outputs["social_signal_snapshots_df"]

print("OSM POI cache rows:", len(osm_city_poi_catalog_df))
print("OSM hotel cache rows:", len(osm_city_hotel_catalog_df))
print("Enriched POI rows:", len(production_enriched_poi_catalog_df))
print("Enrichment audit rows:", len(production_enrichment_audit_df))
display(city_catalog_summary_df)
production_enriched_poi_catalog_df[canonical_poi_columns()].head(12)

OSM POI cache rows: 210
OSM hotel cache rows: 84
Enriched POI rows: 228
Enrichment audit rows: 107


,city,candidate_attractions,mean_utility,total_reviews,data_status,fallback_source,popularity_score,external_poi_score,social_signal_score,route_importance_score,yelp_signal_score,city_value_score,data_confidence,data_uncertainty,data_quality_score,yelp_status,city_must_go_count,city_must_go_score,city_social_score
0,San Diego,30,0.000000,0,multi_source_enriched,multi_source_enriched_catalog,0.860,0.500,0.00,0.52,0.00,0.410,0.480,0.520,0.480,yelp_sparse,0.0,0.0000,0.00
1,Los Angeles,33,0.322110,0,multi_source_enriched,multi_source_enriched_catalog,0.980,0.550,0.99,1.00,0.00,0.751,0.452,0.548,0.452,yelp_sparse,3.0,2.6225,0.99
2,Santa Barbara,40,0.653485,3263,multi_source_enriched,multi_source_enriched_catalog,0.799,0.667,0.88,0.82,0.95,0.818,0.367,0.633,0.367,yelp_sparse,2.0,1.3476,0.88
3,San Luis Obispo,30,0.595776,0,multi_source_enriched,multi_source_enriched_catalog,0.580,0.500,0.00,0.72,0.00,0.346,0.387,0.613,0.387,yelp_sparse,0.0,0.0000,0.00
4,Monterey,33,0.486516,0,multi_source_enriched,multi_source_enriched_catalog,0.700,0.550,0.92,0.78,0.00,0.628,0.376,0.624,0.376,yelp_sparse,1.0,0.7912,0.92
5,Santa Cruz,30,0.482534,0,multi_source_enriched,multi_source_enriched_catalog,0.660,0.500,0.00,0.70,0.00,0.368,0.403,0.597,0.403,yelp_sparse,0.0,0.0000,0.00
6,San Francisco,32,0.570595,0,multi_source_enriched,multi_source_enriched_catalog,0.960,0.533,0.98,1.00,0.00,0.740,0.469,0.531,0.469,yelp_sparse,2.0,1.8718,0.98


,name,city,latitude,longitude,category,source_list,yelp_rating,yelp_review_count,osm_tags,wikidata_id,...,outdoor_intensity,weather_sensitivity,seasonality_risk,park_type,nature_region,interest_fit,park_bonus,interest_adjusted_value,interest_delta,reason_selected
0,Stanford University Main Quad,San Francisco,37.427500,-122.169700,social_must_go:campus,curated_social_must_go,0.0,0.0,,,...,0.000000,0.250000,0.00,,,0.2750,0.0,1.641571,0.0,
1,Santa Barbara County Courthouse,Santa Barbara,34.424201,-119.702328,"Courthouses, Active Life, Event Planning & Ser...",curated_social_must_go|yelp,4.5,251.0,,,...,0.196875,0.377969,0.15,,Santa Barbara,0.3625,0.0,1.512248,0.0,
2,Bixby Creek Bridge Viewpoint,Monterey,36.371500,-121.901800,social_must_go:viewpoint,curated_social_must_go,0.0,0.0,,,...,0.449375,0.542094,0.15,,Monterey,0.1375,0.3,1.502620,0.0,
3,Stearns Wharf,Santa Barbara,34.410014,-119.685550,"Public Services & Government, Landmarks & Hist...",curated_social_must_go|yelp,4.0,297.0,,,...,0.000000,0.250000,0.00,,,0.5000,0.0,1.469428,0.0,
4,Golden Gate Bridge,San Francisco,37.819900,-122.478300,social_must_go:landmark,curated_social_must_go,0.0,0.0,,,...,0.252500,0.414125,0.00,,,0.3875,0.3,1.317181,0.0,
5,Hollywood Walk of Fame,Los Angeles,34.101600,-118.326900,social_must_go:landmark,curated_social_must_go,0.0,0.0,,,...,0.000000,0.250000,0.00,,,0.3375,0.0,1.228090,0.0,
6,Griffith Observatory,Los Angeles,34.118400,-118.300400,social_must_go:viewpoint,curated_social_must_go,0.0,0.0,,,...,0.449375,0.542094,0.15,,Los Angeles,0.3500,0.3,1.155989,0.0,
7,TCL Chinese Theatre,Los Angeles,34.102000,-118.340900,social_must_go:landmark,curated_social_must_go,0.0,0.0,,,...,0.000000,0.250000,0.00,,,0.7000,0.0,1.081067,0.0,
8,Sustainable Wine Tours,Santa Barbara,34.422452,-119.705021,"Wine Tours, Hotels & Travel, Tours, Transporta...",yelp,5.0,358.0,,,...,0.000000,0.250000,0.00,,,0.2750,0.0,0.800367,0.0,
9,SB Buggie,Santa Barbara,34.415540,-119.689705,"Tours, Motorcycle Rental, Scooter Rentals, Kid...",yelp,5.0,231.0,,,...,0.000000,0.250000,0.00,,,0.0625,0.0,0.770178,0.0,


## 14. Hierarchical Multi-City Gurobi Planner


In [17]:
hierarchical_trip_candidates = candidate_plans(config, city_catalog_summary_df)
best_hierarchical_trip, hierarchical_trip_df = solve_hierarchical_trip_with_gurobi(config, city_catalog_summary_df)

assert sum(best_hierarchical_trip["days_by_city"].values()) == int(config.get("trip", "trip_days", 7))
hierarchical_trip_df = hierarchical_trip_df.assign(
    gateway=hierarchical_trip_df.apply(lambda row: f"{row['gateway_start']} -> {row['gateway_end']}", axis=1),
    days_by_city_json=hierarchical_trip_df["days_by_city"].apply(json.dumps),
    drive_hours=hierarchical_trip_df["intercity_drive_minutes"] / 60.0,
).sort_values("objective", ascending=False)
hierarchical_trip_df.to_csv(OUTPUT_DIR / "production_hierarchical_gurobi_plan.csv", index=False)
hierarchical_display_columns = [
    "gateway",
    "city_sequence",
    "overnight_bases",
    "pass_through_cities",
    "days_by_city_json",
    "drive_hours",
    "city_value_component",
    "pass_through_value_component",
    "data_uncertainty_exploration_bonus",
    "base_switch_penalty",
    "objective",
    "solver_status",
]
hierarchical_trip_df[[col for col in hierarchical_display_columns if col in hierarchical_trip_df.columns]].head()

Restricted license - for non-production use only - expires 2027-11-29


,gateway,city_sequence,overnight_bases,pass_through_cities,days_by_city_json,drive_hours,city_value_component,pass_through_value_component,data_uncertainty_exploration_bonus,base_switch_penalty,objective,solver_status
0,San Francisco -> Los Angeles,"[San Francisco, Santa Cruz, Monterey, San Luis...","[San Francisco, Santa Barbara, Los Angeles]","[Santa Cruz, Monterey, San Luis Obispo]","{""San Francisco"": 2, ""Santa Barbara"": 2, ""Los ...",10.283888,4.599258,0.252455,0.05136,0.07,9.254422,gurobi_status_2
1,Los Angeles -> San Francisco,"[Los Angeles, Santa Barbara, San Luis Obispo, ...","[Los Angeles, Santa Barbara, San Francisco]","[San Luis Obispo, Monterey, Santa Cruz]","{""Los Angeles"": 3, ""Santa Barbara"": 2, ""San Fr...",10.283888,4.599258,0.252455,0.05136,0.07,9.254422,gurobi_status_2
2,Los Angeles -> San Francisco,"[Los Angeles, Santa Barbara, San Luis Obispo, ...","[Los Angeles, Santa Barbara, San Francisco]","[San Luis Obispo, Monterey, Santa Cruz]","{""Los Angeles"": 3, ""Santa Barbara"": 1, ""San Fr...",10.283888,4.488431,0.252455,0.05136,0.07,9.185198,gurobi_status_2
3,San Francisco -> Los Angeles,"[San Francisco, Santa Cruz, Monterey, San Luis...","[San Francisco, Santa Barbara, Los Angeles]","[Santa Cruz, Monterey, San Luis Obispo]","{""San Francisco"": 3, ""Santa Barbara"": 1, ""Los ...",10.283888,4.488431,0.252455,0.05136,0.07,9.185198,gurobi_status_2
4,San Francisco -> Los Angeles,"[San Francisco, Santa Cruz, Monterey, San Luis...","[San Francisco, Santa Barbara, Los Angeles]","[Santa Cruz, Monterey, San Luis Obispo]","{""San Francisco"": 1, ""Santa Barbara"": 3, ""Los ...",10.283888,4.602446,0.252455,0.05136,0.07,9.181978,gurobi_status_2


## 15. Approximate Budget Range


In [18]:
budget_estimate_df = estimate_budget_range(
    trip=best_hierarchical_trip,
    hotels_df=hotels_df,
    osm_city_hotel_catalog_df=osm_city_hotel_catalog_df,
    output_dir=OUTPUT_DIR,
    config=config,
    primary_city=CITY,
)
_budget_bounds = budget_bounds_from_df(budget_estimate_df)
soft_budget = _budget_bounds.soft_budget
hard_budget = _budget_bounds.hard_budget
trip_length_comparison_df = compare_trip_lengths(
    config=config,
    city_summary_df=city_catalog_summary_df,
    hotels_df=hotels_df,
    osm_city_hotel_catalog_df=osm_city_hotel_catalog_df,
    output_dir=OUTPUT_DIR,
    primary_city=CITY,
)
budget_estimate_df

,component,low,expected,high
0,hotel,764.40,980.00,1323.00
1,attractions,70.56,196.00,384.16
2,food,337.68,504.00,806.40
3,fuel_parking_local_transport,134.42,244.47,401.99
4,buffered_total,1411.63,2213.14,3556.97
5,soft_budget,2000.00,2000.00,2000.00
6,hard_budget,3556.97,3556.97,3556.97


## 16. Bandit-Guided Small Gurobi Route Benchmark


In [19]:
ROUTE_SEARCH_STRATEGIES = route_search_strategies_from_config(config)

hybrid_bandit_arms_df = build_bandit_arms(
    hierarchical_candidates=hierarchical_trip_candidates,
    enriched_df=production_enriched_poi_catalog_df,
    budget_df=budget_estimate_df,
    config=config,
)
hybrid_bandit_log_df, hybrid_bandit_summary_df = run_bandit_search(hybrid_bandit_arms_df, config)
bandit_stress_runs_df, bandit_stress_summary_df = run_bandit_gurobi_stress_benchmark(
    arms_df=hybrid_bandit_arms_df,
    enriched_df=production_enriched_poi_catalog_df,
    budget_df=budget_estimate_df,
    config=config,
)
hybrid_bandit_log_df.to_csv(OUTPUT_DIR / "production_bandit_gurobi_runs.csv", index=False)
hybrid_bandit_summary_df.to_csv(OUTPUT_DIR / "production_hybrid_bandit_optimization_summary.csv", index=False)
bandit_stress_runs_df.to_csv(OUTPUT_DIR / "production_bandit_stress_runs.csv", index=False)
bandit_stress_summary_df.to_csv(OUTPUT_DIR / "production_bandit_stress_summary.csv", index=False)
bandit_stress_summary_df.to_csv(OUTPUT_DIR / "production_solver_scalability_summary.csv", index=False)

mmr_candidate_log_df = bandit_stress_runs_df.attrs.get("mmr_candidate_log_df", pd.DataFrame())
mmr_candidate_log_df.to_csv(OUTPUT_DIR / "production_mmr_candidate_log.csv", index=False)
route_component_columns = [
    "arm_id",
    "route_search_strategy",
    "selected_pois",
    "gurobi_oracle_reward",
    "combined_reward",
    "route_feasible",
    "submodular_diversity",
    "total_travel_minutes",
    "total_visit_minutes",
    "total_detour_minutes",
    "total_cost",
    "total_weather_risk",
    "epsilon_time_budget",
    "epsilon_cost_budget",
    "epsilon_detour_minutes",
    "epsilon_weather_risk",
    "epsilon_min_diversity",
    "solver_status",
    "optimality_gap",
]
bandit_stress_runs_df[[col for col in route_component_columns if col in bandit_stress_runs_df.columns]].to_csv(
    OUTPUT_DIR / "production_route_objective_components.csv", index=False
)
pd.DataFrame(
    [
        diversity_audit_row(production_enriched_poi_catalog_df, "enriched_catalog"),
    ]
).to_csv(OUTPUT_DIR / "production_diversity_audit.csv", index=False)

print("Bandit arms:", len(hybrid_bandit_arms_df))
print("Stress benchmark rows:", len(bandit_stress_runs_df))
display(bandit_stress_summary_df.head(8))
hybrid_bandit_summary_df.head(8)

Bandit arms: 2605
Stress benchmark rows: 81


,episodes,candidate_size,route_search_strategy,runs,mean_combined_reward,best_combined_reward,mean_runtime_seconds,feasible_rate,mean_selected_poi_count,mean_optimality_gap
10,40,8,fastest_low_cost,3,10.539292,10.539951,0.078559,1.0,4.0,0.0
19,80,8,fastest_low_cost,3,10.539486,10.539722,0.074757,1.0,4.0,0.0
1,20,8,fastest_low_cost,3,10.532328,10.534068,0.084482,1.0,4.0,0.0
13,40,12,fastest_low_cost,3,10.525102,10.528480,0.220451,1.0,4.0,0.0
22,80,12,fastest_low_cost,3,10.527055,10.528465,0.199068,1.0,4.0,0.0
4,20,12,fastest_low_cost,3,10.515606,10.521008,0.251702,1.0,4.0,0.0
16,40,16,fastest_low_cost,3,10.492190,10.493752,0.549572,1.0,4.0,0.0
25,80,16,fastest_low_cost,3,10.486527,10.492750,0.604347,1.0,4.0,0.0


,arm_id,gateway_start,gateway_end,city_sequence,days_by_city,route_search_strategy,candidate_bundle,bandit_role,optimizer_role,small_optimizer_calls,...,soft_budget_penalty,poi_value_signal,social_signal,must_go_signal,corridor_signal,detour_minutes_signal,candidate_bundle_size,reward_prior,pulls,posterior_mean_reward
171,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 1, 'Santa Barbara': 3, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.201612,40,9.277092
176,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 2, 'Santa Barbara': 2, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.370388,40,9.277092
181,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 2, 'Santa Barbara': 3, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.368854,40,9.277092
186,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 1, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.388800,40,9.277092
191,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 2, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.458024,40,9.277092
196,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 3, 'San Fr...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,9.385580,40,9.277092
201,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 1, 'San Luis Obispo': 3, 'San ...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,7.635029,40,9.277092
206,Los Angeles->San Francisco|fastest_low_cost,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 2, 'San Luis Obispo': 2, 'San ...",fastest_low_cost,fastest_low_cost,route_finding_candidate_search,fast_local_route_oracle,3,...,0.0,0.917122,0.2996,0.265324,0.892876,4.8208,25,8.276036,40,9.277092


## 18. Canonical Three-Method Hierarchical Comparison

This section replaces the old standalone local demo path. All production comparisons now share the same hierarchical contract: choose gateway/cities/day allocation first, then solve the deeper city/day routes.

- **Hierarchical Gurobi:** Gurobi master allocation + local Gurobi day routes.
- **Hierarchical Greedy:** greedy master allocation + greedy local day routes.
- **Hybrid Bandit + Gurobi Repair:** bandit strategy selection + small Gurobi route repair.


In [20]:
import itinerary_system.experiment_runner as experiment_runner_module

experiment_runner_module = importlib.reload(experiment_runner_module)

comparison_context = dict(globals())
comparison_context["OUTPUT_DIR"] = OUTPUT_DIR
comparison_context["FIGURE_DIR"] = FIGURE_DIR
comparison_context["PROJECT_ROOT"] = PROJECT_ROOT

production_method_comparison_df, production_method_route_stops_df = (
    experiment_runner_module.export_method_comparison_dashboard(
        context=comparison_context,
        config=config,
        output_dir=OUTPUT_DIR,
        primary_city=CITY,
    )
)
production_trip_length_comparison_df, production_trip_length_route_stops_df = (
    experiment_runner_module.export_trip_length_comparison_dashboard(
        context=comparison_context,
        config=config,
        output_dir=OUTPUT_DIR,
        primary_city=CITY,
    )
)

metric_columns = [
    "method",
    "method_display_name",
    "allocation_solver",
    "local_route_solver",
    "total_utility",
    "must_go_count",
    "diversity_score",
    "total_travel_time",
    "total_cost",
    "comparison_score",
    "status",
]
metric_columns = [column for column in metric_columns if column in production_method_comparison_df.columns]

print("Canonical hierarchical method comparison artifacts:")
print("  production_method_comparison.csv:", production_method_comparison_df.shape)
print("  production_method_route_stops.csv:", production_method_route_stops_df.shape)
print("  production_trip_length_comparison.csv:", production_trip_length_comparison_df.shape)
print("  production_trip_length_route_stops.csv:", production_trip_length_route_stops_df.shape)
print("\nMetric definitions:")
print("  U_m = sum selected POI values")
print("  T_m = sum route/intercity travel minutes")
print("  C_m = hotel + transport reserve + selected POI costs")
print("  M_m = count of social must-go stops")
print("  H_m = normalized Shannon diversity over selected POI categories")
print("  Score_m = 0.40*U_norm + 0.15*M_norm + 0.15*H + 0.15*T_eff + 0.10*C_eff + 0.05*W_eff")

display(production_method_comparison_df[metric_columns])
display(production_trip_length_comparison_df.head())

Canonical hierarchical method comparison artifacts:
  production_method_comparison.csv: (3, 40)
  production_method_route_stops.csv: (47, 61)
  production_trip_length_comparison.csv: (3, 17)
  production_trip_length_route_stops.csv: (67, 61)

Metric definitions:
  U_m = sum selected POI values
  T_m = sum route/intercity travel minutes
  C_m = hotel + transport reserve + selected POI costs
  M_m = count of social must-go stops
  H_m = normalized Shannon diversity over selected POI categories
  Score_m = 0.40*U_norm + 0.15*M_norm + 0.15*H + 0.15*T_eff + 0.10*C_eff + 0.05*W_eff


,method,method_display_name,allocation_solver,local_route_solver,total_utility,must_go_count,diversity_score,total_travel_time,total_cost,comparison_score,status
0,hierarchical_gurobi_pipeline,Hierarchical Gurobi Pipeline,gurobi_master,legacy_local_gurobi,10.756862,5,0.938357,1854.023943,1993.803333,0.411320,OPTIMAL
1,hierarchical_greedy_baseline,Hierarchical Greedy Baseline,greedy_master,greedy_local_route,14.958645,6,0.941618,1922.820817,2428.470000,0.831988,HEURISTIC
2,hierarchical_bandit_gurobi_repair,Hierarchical + Bandit + Small Gurobi Repair,bandit_master,small_gurobi_repair,14.214890,6,0.948317,2056.946841,2041.803333,0.747790,HEURISTIC_FALLBACK


,comparison_type,comparison_label,trip_days,gateway_start,gateway_end,city_sequence,days_by_city,intercity_drive_minutes,intercity_drive_hours,estimated_budget,budget_source,objective,posterior_reward,selected_strategy,solver_status,status,notes
0,trip_length,Trip Length · 7-Day Hybrid Bandit + Small Gurobi,7,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 2, 'San Fr...",617.033264,10.283888,2213.14,buffered_total,9.254422,NaN,hierarchical_bandit_gurobi_repair:fastest_low_...,gurobi_status_2,HEURISTIC_FALLBACK,Trip-length comparison uses top-level hierarch...
1,trip_length,Trip Length · 9-Day Hybrid Bandit + Small Gurobi,9,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 3, 'San Fr...",617.033264,10.283888,2864.04,buffered_total,10.682271,NaN,hierarchical_bandit_gurobi_repair:fastest_low_...,gurobi_status_2,HEURISTIC_FALLBACK,Trip-length comparison uses top-level hierarch...
2,trip_length,Trip Length · 12-Day Hybrid Bandit + Small Gurobi,12,Los Angeles,San Francisco,Los Angeles -> Santa Barbara -> San Luis Obisp...,"{'Los Angeles': 3, 'Santa Barbara': 3, 'Monter...",617.033264,10.283888,3788.64,buffered_total,12.351019,NaN,hierarchical_bandit_gurobi_repair:fastest_low_...,gurobi_status_2,HEURISTIC_FALLBACK,Trip-length comparison uses top-level hierarch...


## 20. Production Experiment Table

The main rows below share `data_source_scope=open_enriched_catalog`. Method-level production comparison is generated in the canonical three-method section above.


In [21]:
def normalized_shannon_diversity(labels):
    values = pd.Series(labels).dropna().astype(str)
    if values.empty:
        return 0.0
    counts = values.value_counts().to_numpy(dtype=float)
    probabilities = counts / counts.sum()
    entropy = -np.sum(probabilities * np.log(probabilities))
    return float(entropy / np.log(len(counts))) if len(counts) > 1 else 0.0


def approximate_route_minutes(frame):
    if frame.empty or len(frame) < 2:
        return 0.0
    points = frame[["latitude", "longitude"]].astype(float).values.tolist()
    return float(
        sum(geodesic(tuple(left), tuple(right)).km * 1.25 / 38.0 * 60.0 for left, right in zip(points[:-1], points[1:]))
    )


def enriched_metric_row(strategy, frame, *, route_feasible=True, runtime_seconds=np.nan, extra=None):
    frame = frame.copy()
    extra = extra or {}
    return {
        "strategy": strategy,
        "profile": "balanced",
        "data_source_scope": "open_enriched_catalog",
        "total_utility": float(
            pd.to_numeric(frame.get("final_poi_value", pd.Series(dtype=float)), errors="coerce").fillna(0).sum()
        ),
        "number_attractions": int(len(frame)),
        "total_waiting_time": np.nan,
        "within_city_travel_time": approximate_route_minutes(frame),
        "intercity_driving_time": 0.0,
        "budget_used": float(
            budget_estimate_df.loc[budget_estimate_df["component"].eq("attractions"), "expected"].iloc[0]
        )
        if not budget_estimate_df.empty
        else np.nan,
        "route_feasibility": bool(route_feasible),
        "diversity_score": normalized_shannon_diversity(frame.get("category", pd.Series(dtype=str))),
        "runtime_seconds": runtime_seconds,
        "optimality_gap": np.nan,
        "data_quality_score": float(
            pd.to_numeric(frame.get("data_confidence", pd.Series(dtype=float)), errors="coerce").fillna(0.15).mean()
        )
        if not frame.empty
        else 0.15,
        "gateway_start": np.nan,
        "gateway_end": np.nan,
        "days_by_city": None,
        "route_search_strategy": None,
        "small_optimizer_calls": np.nan,
        **extra,
    }


def build_enriched_ranking_baseline(max_stops=None):
    max_stops = max_stops or int(config.get("trip", "trip_days", 7)) * 3
    return production_enriched_poi_catalog_df.sort_values(["final_poi_value", "data_confidence"], ascending=False).head(
        max_stops
    )


def build_enriched_greedy_baseline(max_stops=None):
    max_stops = max_stops or int(config.get("trip", "trip_days", 7)) * 3
    pool = production_enriched_poi_catalog_df.copy().sort_values("final_poi_value", ascending=False).head(max_stops * 3)
    if pool.empty:
        return pool
    selected = [pool.index[0]]
    remaining = set(pool.index.tolist()) - set(selected)
    while remaining and len(selected) < max_stops:
        current = production_enriched_poi_catalog_df.loc[selected[-1], ["latitude", "longitude"]].astype(float).tolist()
        next_idx = max(
            remaining,
            key=lambda idx: (
                float(production_enriched_poi_catalog_df.loc[idx, "final_poi_value"])
                - 0.01
                * geodesic(
                    tuple(current),
                    tuple(
                        production_enriched_poi_catalog_df.loc[idx, ["latitude", "longitude"]].astype(float).tolist()
                    ),
                ).km
            ),
        )
        selected.append(next_idx)
        remaining.remove(next_idx)
    return production_enriched_poi_catalog_df.loc[selected].copy()


ranking_frame = build_enriched_ranking_baseline()
greedy_frame = build_enriched_greedy_baseline()
experiment_rows = []
for utility_strategy, utility_column in [
    ("mcda_weighted", "utility_mcda_weighted"),
    ("topsis", "utility_topsis"),
    ("bayesian_ucb", "utility_bayesian_ucb"),
]:
    if utility_column in production_enriched_poi_catalog_df.columns:
        utility_frame = production_enriched_poi_catalog_df.sort_values(
            [utility_column, "data_confidence"], ascending=False
        ).head(int(config.get("trip", "trip_days", 7)) * 3)
        utility_row = enriched_metric_row(
            utility_strategy, utility_frame, extra={"utility_score_column": utility_column}
        )
        utility_row["total_utility"] = float(
            pd.to_numeric(utility_frame[utility_column], errors="coerce").fillna(0.0).sum()
        )
        experiment_rows.append(utility_row)

experiment_rows.extend(
    [
        enriched_metric_row("ranking_baseline", ranking_frame),
        enriched_metric_row("greedy_route_baseline", greedy_frame),
    ]
)

epsilon_candidates = production_enriched_poi_catalog_df.sort_values("final_poi_value", ascending=False).head(
    int(config.get("optimization", "max_pois_per_day", 4)) * 4
)
epsilon_route_result = solve_enriched_route_with_gurobi(
    epsilon_candidates,
    config,
    candidate_size=len(epsilon_candidates),
)
pd.DataFrame([epsilon_route_result]).to_csv(OUTPUT_DIR / "production_pareto_epsilon_runs.csv", index=False)
pd.DataFrame(
    [
        diversity_audit_row(production_enriched_poi_catalog_df, "enriched_catalog"),
        diversity_audit_row(epsilon_candidates, "epsilon_gurobi_candidate_pool"),
    ]
).to_csv(OUTPUT_DIR / "production_diversity_audit.csv", index=False)
experiment_rows.append(
    {
        "strategy": "epsilon_gurobi",
        "profile": "balanced",
        "data_source_scope": "open_enriched_catalog",
        "total_utility": float(epsilon_route_result.get("objective_value", np.nan)),
        "number_attractions": int(len(epsilon_route_result.get("selected_pois", []))),
        "total_waiting_time": np.nan,
        "within_city_travel_time": float(epsilon_route_result.get("total_travel_minutes", np.nan)),
        "intercity_driving_time": 0.0,
        "budget_used": float(epsilon_route_result.get("total_cost", np.nan)),
        "route_feasibility": bool(epsilon_route_result.get("feasible", False)),
        "diversity_score": float(epsilon_route_result.get("diversity_score", np.nan)),
        "runtime_seconds": float(epsilon_route_result.get("runtime_seconds", np.nan)),
        "optimality_gap": epsilon_route_result.get("optimality_gap", np.nan),
        "data_quality_score": float(
            pd.to_numeric(
                production_enriched_poi_catalog_df.get("data_confidence", pd.Series(dtype=float)), errors="coerce"
            )
            .fillna(0.15)
            .mean()
        ),
        "gateway_start": np.nan,
        "gateway_end": np.nan,
        "days_by_city": None,
        "route_search_strategy": "epsilon_constraint_route",
        "small_optimizer_calls": 1,
        "solver_status": epsilon_route_result.get("solver_status"),
    }
)


experiment_rows.append(
    {
        "strategy": "hierarchical_gurobi",
        "profile": "balanced",
        "data_source_scope": "open_enriched_catalog",
        "total_utility": best_hierarchical_trip["objective"],
        "number_attractions": np.nan,
        "total_waiting_time": np.nan,
        "within_city_travel_time": np.nan,
        "intercity_driving_time": best_hierarchical_trip["intercity_drive_minutes"],
        "budget_used": float(
            budget_estimate_df.loc[budget_estimate_df["component"].eq("buffered_total"), "expected"].iloc[0]
        ),
        "route_feasibility": sum(best_hierarchical_trip["days_by_city"].values())
        == int(config.get("trip", "trip_days", 7)),
        "diversity_score": normalized_shannon_diversity(best_hierarchical_trip["days_by_city"].keys()),
        "runtime_seconds": np.nan,
        "optimality_gap": np.nan,
        "data_quality_score": float(city_catalog_summary_df["data_quality_score"].mean()),
        "gateway_start": best_hierarchical_trip["gateway_start"],
        "gateway_end": best_hierarchical_trip["gateway_end"],
        "days_by_city": json.dumps(best_hierarchical_trip["days_by_city"]),
        "route_search_strategy": None,
        "small_optimizer_calls": np.nan,
    }
)

if not bandit_stress_runs_df.empty:
    best_hybrid = bandit_stress_runs_df.sort_values("combined_reward", ascending=False).iloc[0]
    experiment_rows.append(
        {
            "strategy": "hybrid_bandit_gurobi",
            "profile": "balanced",
            "data_source_scope": "open_enriched_catalog",
            "total_utility": float(best_hybrid.get("combined_reward", np.nan)),
            "number_attractions": int(best_hybrid.get("selected_poi_count", 0)),
            "total_waiting_time": np.nan,
            "within_city_travel_time": float(best_hybrid.get("total_travel_minutes", np.nan)),
            "intercity_driving_time": np.nan,
            "budget_used": float(best_hybrid.get("estimated_total_cost", np.nan)),
            "route_feasibility": bool(best_hybrid.get("route_feasible", True)),
            "diversity_score": np.nan,
            "runtime_seconds": float(best_hybrid.get("runtime_seconds", np.nan)),
            "optimality_gap": best_hybrid.get("optimality_gap", np.nan),
            "data_quality_score": float(city_catalog_summary_df["data_quality_score"].mean()),
            "gateway_start": best_hybrid.get("gateway_start"),
            "gateway_end": best_hybrid.get("gateway_end"),
            "days_by_city": None,
            "route_search_strategy": best_hybrid.get("route_search_strategy"),
            "small_optimizer_calls": int(best_hybrid.get("small_optimizer_calls", 0)),
        }
    )
else:
    best_hybrid = hybrid_bandit_summary_df.iloc[0]
    experiment_rows.append(
        {
            "strategy": "hybrid_bandit_gurobi",
            "profile": "balanced",
            "data_source_scope": "open_enriched_catalog",
            "total_utility": float(best_hybrid["posterior_mean_reward"]),
            "number_attractions": np.nan,
            "total_waiting_time": np.nan,
            "within_city_travel_time": np.nan,
            "intercity_driving_time": np.nan,
            "budget_used": float(best_hybrid.get("estimated_total_cost", np.nan)),
            "route_feasibility": bool(best_hybrid.get("hard_budget_feasible", True)),
            "diversity_score": normalized_shannon_diversity(best_hybrid["days_by_city"].keys()),
            "runtime_seconds": float(best_hybrid["estimated_solver_minutes"]) * 60.0,
            "optimality_gap": np.nan,
            "data_quality_score": float(city_catalog_summary_df["data_quality_score"].mean()),
            "gateway_start": best_hybrid["gateway_start"],
            "gateway_end": best_hybrid["gateway_end"],
            "days_by_city": json.dumps(best_hybrid["days_by_city"]),
            "route_search_strategy": best_hybrid["route_search_strategy"],
            "small_optimizer_calls": int(best_hybrid["small_optimizer_calls"]),
        }
    )

production_experiment_summary = pd.DataFrame(experiment_rows)
production_experiment_summary["optimized_total_utility"] = production_experiment_summary["total_utility"]
production_experiment_summary["optimized_total_wait"] = production_experiment_summary["total_waiting_time"]
production_experiment_summary["optimized_total_cost"] = production_experiment_summary["budget_used"]
production_experiment_summary["intercity_drive_hours"] = production_experiment_summary["intercity_driving_time"] / 60.0
production_experiment_summary.to_csv(OUTPUT_DIR / "production_experiment_summary.csv", index=False)
production_experiment_summary.to_csv(OUTPUT_DIR / "production_model_comparison.csv", index=False)

production_experiment_runs = bandit_stress_runs_df.assign(
    experiment_id="hybrid_bandit_gurobi_stress",
    strategy="hybrid_bandit_gurobi",
    data_source_scope="open_enriched_catalog",
)
production_experiment_runs.to_csv(OUTPUT_DIR / "production_experiment_runs.csv", index=False)
hybrid_bandit_log_df.to_csv(OUTPUT_DIR / "production_hybrid_bandit_optimization_log.csv", index=False)
hybrid_bandit_summary_df.to_csv(OUTPUT_DIR / "production_hybrid_bandit_optimization_summary.csv", index=False)

production_experiment_summary

,strategy,profile,data_source_scope,total_utility,number_attractions,total_waiting_time,within_city_travel_time,intercity_driving_time,budget_used,route_feasibility,...,gateway_end,days_by_city,route_search_strategy,small_optimizer_calls,utility_score_column,solver_status,optimized_total_utility,optimized_total_wait,optimized_total_cost,intercity_drive_hours
0,mcda_weighted,balanced,open_enriched_catalog,9.538054,21.0,NaN,6073.381329,0.000000,196.0000,True,...,NaN,None,None,NaN,utility_mcda_weighted,NaN,9.538054,NaN,196.0000,0.000000
1,topsis,balanced,open_enriched_catalog,9.081162,21.0,NaN,4611.586311,0.000000,196.0000,True,...,NaN,None,None,NaN,utility_topsis,NaN,9.081162,NaN,196.0000,0.000000
2,bayesian_ucb,balanced,open_enriched_catalog,20.213036,21.0,NaN,8243.339718,0.000000,196.0000,True,...,NaN,None,None,NaN,utility_bayesian_ucb,NaN,20.213036,NaN,196.0000,0.000000
3,ranking_baseline,balanced,open_enriched_catalog,20.213036,21.0,NaN,8243.339718,0.000000,196.0000,True,...,NaN,None,None,NaN,NaN,NaN,20.213036,NaN,196.0000,0.000000
4,greedy_route_baseline,balanced,open_enriched_catalog,16.185046,21.0,NaN,602.591024,0.000000,196.0000,True,...,NaN,None,None,NaN,NaN,NaN,16.185046,NaN,196.0000,0.000000
5,epsilon_gurobi,balanced,open_enriched_catalog,3.868123,4.0,NaN,338.419536,0.000000,28.0000,True,...,NaN,None,epsilon_constraint_route,1.0,NaN,gurobi_status_2,3.868123,NaN,28.0000,0.000000
6,hierarchical_gurobi,balanced,open_enriched_catalog,9.254422,NaN,NaN,NaN,617.033264,2213.1400,True,...,San Francisco,"{""Los Angeles"": 3, ""Santa Barbara"": 2, ""San Fr...",None,NaN,NaN,NaN,9.254422,NaN,2213.1400,10.283888
7,hybrid_bandit_gurobi,balanced,open_enriched_catalog,10.539951,4.0,NaN,334.869866,NaN,1947.5632,True,...,San Francisco,None,fastest_low_cost,3.0,NaN,NaN,10.539951,NaN,1947.5632,NaN


## 21. Interactive HTML Trip Result


In [22]:
import importlib
import subprocess
import sys

import blueprint_trip_map
from IPython.display import HTML, display

import itinerary_system.map_renderer as map_renderer

blueprint_trip_map = importlib.reload(blueprint_trip_map)
map_renderer = importlib.reload(map_renderer)

print("Using blueprint:", blueprint_trip_map.__file__)
print("Using map renderer:", map_renderer.__file__)

map_context = dict(globals())
map_context["FORCE_REBUILD_MAP_DASHBOARD_DATA"] = bool(globals().get("FORCE_REBUILD_MAP_DASHBOARD_DATA", False))
map_context["MAP_FOCUS_MODE"] = "full_trip"

# Canonical production-map defaults: Balanced-only first view; details stay checkbox-toggleable.
map_context["MAP_ROUTE_ONLY_DEBUG_VIEW"] = False
map_context["MAP_BALANCED_ONLY_DEFAULT_VIEW"] = True
map_context["MAP_REFRESH_ROAD_GEOMETRY"] = True
map_context["SHOW_FULL_SCENE_DEFAULT"] = False
map_context["SHOW_CONTEXT_ROUTES_BY_DEFAULT"] = False
map_context["SHOW_COMPARISON_LAYERS_BY_DEFAULT"] = False
map_context["SHOW_TRAVELER_OVERVIEWS_BY_DEFAULT"] = False
map_context["SHOW_SELECTED_RESULT_BY_DEFAULT"] = False
map_context["SHOW_SOCIAL_CANDIDATES_BY_DEFAULT"] = False
map_context["SHOW_CANDIDATE_HOTELS_BY_DEFAULT"] = False

production_trip_map, production_day_plan_df, production_html_map_path = map_renderer.build_map(
    map_context,
    config=config,
    output_path=FIGURE_DIR / "production_hierarchical_trip_map.html",
)

html_uri = production_html_map_path.resolve().as_uri()

print("Map source files:")
for artifact in [
    OUTPUT_DIR / "production_method_comparison.csv",
    OUTPUT_DIR / "production_method_route_stops.csv",
    OUTPUT_DIR / "production_trip_length_comparison.csv",
    OUTPUT_DIR / "production_trip_length_route_stops.csv",
    OUTPUT_DIR / "production_day_plan.csv",
    OUTPUT_DIR / "production_day_plan_profiles.csv",
    OUTPUT_DIR / "production_map_route_debug.csv",
]:
    if artifact.exists():
        rows = pd.read_csv(artifact).shape[0]
        print(f"  {artifact.name}: {rows} rows -> {artifact}")
    else:
        print(f"  MISSING: {artifact}")

validation = subprocess.run(
    [sys.executable, str(NOTEBOOK_DIR / "validate_trip_map.py"), "--strict"],
    cwd=PROJECT_ROOT,
    capture_output=True,
    text=True,
)
print(validation.stdout)
if validation.returncode != 0:
    print(validation.stderr)
    raise RuntimeError(
        "Map validation failed; inspect production_map_validation_report.csv and production_map_validation_diagnosis.json"
    )

display(
    HTML(f"""
<iframe
    src="{html_uri}?v={int(time.time())}"
    width="100%"
    height="850"
    style="border:1px solid #ddd; border-radius:8px;"
></iframe>
""")
)

print("Open HTML:", production_html_map_path)

Using blueprint: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/notebook/blueprint_trip_map.py
Using map renderer: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/src/itinerary_system/map_renderer.py
Map dashboard context: trip_days=7, budget=2213.14, local_gurobi_demo_excluded=True
Keeping existing production comparison dashboard artifacts.
Map source files:
  production_method_comparison.csv: 3 rows -> /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/results/outputs/production_method_comparison.csv
  production_method_route_stops.csv: 47 rows -> /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/results/outputs/production_method_route_stops.csv
  production_trip_length_comparison.csv: 3 rows -> /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/results/outputs/production_trip_length_comparison.csv


Open HTML: /Users/chenyixin/Documents/IE 5533/Project/weather-aware-travel-itinerary-optimization/results/figures/production_hierarchical_trip_map.html


## 24. [PROTOTYPE] FastAPI, PostgreSQL, AWS, and Streamlit Architecture


In [23]:
recommend_request_example = {
    "state": "California",
    "trip_days": 7,
    "start_city_options": ["Los Angeles", "San Francisco"],
    "end_city_options": ["Los Angeles", "San Francisco"],
    "profile": "balanced",
    "budget": 2000,
    "strategy": "hybrid_bandit_optimization_search",
    "use_social_signals": True,
}

recommend_response_contract = {
    "trip_id": "trip_ca_7d_001",
    "gateway_start": best_hierarchical_trip["gateway_start"],
    "gateway_end": best_hierarchical_trip["gateway_end"],
    "city_sequence": best_hierarchical_trip["city_sequence"],
    "days_by_city": best_hierarchical_trip["days_by_city"],
    "intercity_drive_hours": round(best_hierarchical_trip["intercity_drive_minutes"] / 60, 2),
    "city_itineraries": "filled by small city-level Gurobi route-seeking calls selected by the bandit layer",
    "feasibility_status": "prototype_feasible",
}

print(json.dumps(recommend_request_example, indent=2))
print(json.dumps(recommend_response_contract, indent=2))

{
  "state": "California",
  "trip_days": 7,
  "start_city_options": [
    "Los Angeles",
    "San Francisco"
  ],
  "end_city_options": [
    "Los Angeles",
    "San Francisco"
  ],
  "profile": "balanced",
  "budget": 2000,
  "strategy": "hybrid_bandit_optimization_search",
  "use_social_signals": true
}
{
  "trip_id": "trip_ca_7d_001",
  "gateway_start": "Los Angeles",
  "gateway_end": "San Francisco",
  "city_sequence": [
    "Los Angeles",
    "Santa Barbara",
    "San Luis Obispo",
    "Monterey",
    "Santa Cruz",
    "San Francisco"
  ],
  "days_by_city": {
    "Los Angeles": 3,
    "Santa Barbara": 2,
    "San Francisco": 2
  },
  "intercity_drive_hours": 10.28,
  "city_itineraries": "filled by small city-level Gurobi route-seeking calls selected by the bandit layer",
  "feasibility_status": "prototype_feasible"
}


## 25. PostgreSQL Schema


In [24]:
postgres_schema_sql = """
CREATE TABLE cities (
    city_id SERIAL PRIMARY KEY,
    city_name TEXT NOT NULL,
    state TEXT NOT NULL,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    data_status TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE attractions (
    attraction_id TEXT PRIMARY KEY,
    city_id INTEGER REFERENCES cities(city_id),
    source TEXT,
    source_business_id TEXT,
    name TEXT NOT NULL,
    category TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    rating_score DOUBLE PRECISION,
    review_count INTEGER,
    utility DOUBLE PRECISION,
    waiting_final DOUBLE PRECISION,
    visit_duration_sim DOUBLE PRECISION,
    cost DOUBLE PRECISION
);

CREATE TABLE trip_requests (
    trip_id TEXT PRIMARY KEY,
    request_json JSONB NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE itinerary_results (
    itinerary_id TEXT PRIMARY KEY,
    trip_id TEXT REFERENCES trip_requests(trip_id),
    strategy TEXT,
    solver_name TEXT,
    objective_value DOUBLE PRECISION,
    relative_gap DOUBLE PRECISION,
    feasibility_status TEXT,
    result_json JSONB,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE city_day_allocations (
    allocation_id SERIAL PRIMARY KEY,
    itinerary_id TEXT REFERENCES itinerary_results(itinerary_id),
    city_name TEXT,
    allocated_days INTEGER
);

CREATE TABLE itinerary_stops (
    stop_id SERIAL PRIMARY KEY,
    itinerary_id TEXT REFERENCES itinerary_results(itinerary_id),
    global_day INTEGER,
    city_name TEXT,
    stop_order INTEGER,
    attraction_name TEXT,
    pred_wait_minutes DOUBLE PRECISION,
    visit_minutes DOUBLE PRECISION,
    cost DOUBLE PRECISION
);

CREATE TABLE intercity_legs (
    leg_id SERIAL PRIMARY KEY,
    itinerary_id TEXT REFERENCES itinerary_results(itinerary_id),
    from_city TEXT,
    to_city TEXT,
    drive_minutes DOUBLE PRECISION
);

CREATE TABLE experiments (
    experiment_id TEXT PRIMARY KEY,
    name TEXT,
    description TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE experiment_runs (
    run_id TEXT PRIMARY KEY,
    experiment_id TEXT REFERENCES experiments(experiment_id),
    strategy TEXT,
    profile TEXT,
    run_json JSONB,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE metric_results (
    metric_id SERIAL PRIMARY KEY,
    run_id TEXT REFERENCES experiment_runs(run_id),
    metric_name TEXT,
    metric_value DOUBLE PRECISION
);

CREATE TABLE social_signal_snapshots (
    snapshot_id SERIAL PRIMARY KEY,
    attraction_id TEXT REFERENCES attractions(attraction_id),
    source_name TEXT,
    social_popularity_score DOUBLE PRECISION,
    social_momentum_score DOUBLE PRECISION,
    crowding_risk_score DOUBLE PRECISION,
    raw_posts_stored BOOLEAN DEFAULT FALSE,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);
"""
print(postgres_schema_sql[:1500] + "\n...")


CREATE TABLE cities (
    city_id SERIAL PRIMARY KEY,
    city_name TEXT NOT NULL,
    state TEXT NOT NULL,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    data_status TEXT,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE attractions (
    attraction_id TEXT PRIMARY KEY,
    city_id INTEGER REFERENCES cities(city_id),
    source TEXT,
    source_business_id TEXT,
    name TEXT NOT NULL,
    category TEXT,
    latitude DOUBLE PRECISION,
    longitude DOUBLE PRECISION,
    rating_score DOUBLE PRECISION,
    review_count INTEGER,
    utility DOUBLE PRECISION,
    waiting_final DOUBLE PRECISION,
    visit_duration_sim DOUBLE PRECISION,
    cost DOUBLE PRECISION
);

CREATE TABLE trip_requests (
    trip_id TEXT PRIMARY KEY,
    request_json JSONB NOT NULL,
    created_at TIMESTAMP DEFAULT CURRENT_TIMESTAMP
);

CREATE TABLE itinerary_results (
    itinerary_id TEXT PRIMARY KEY,
    trip_id TEXT REFERENCES trip_requests(trip_id),
    strategy TEXT,
    

## 26. AWS EC2 Deployment Plan


## 27. Final Publishable and Resume Framing
